# 6. Run PyNNLF: SA BESS 44hh Clean Cohort

Databricks full-run version. This notebook runs the complete clean SA BESS 44-household experiment block in a separate Databricks output folder, so it does not mix with the earlier partial local UNSW laptop run.

Expected experiment count: `3 datasets x 1 horizon x 12 models = 36`.


## 1. Discover Paths

The path logic is relative and works from Databricks Repos/Workspace or from a local checkout. It finds the repository root from `pyproject.toml` and the publication project from `publication/journal_article_1`.


In [ ]:
from pathlib import Path
import os
import sys


def databricks_notebook_candidates():
    candidates = []
    try:
        ctx = dbutils.notebook.entry_point.getDbutils().notebook().getContext()  # noqa: F821
        nb_path = ctx.notebookPath().get()
    except Exception:
        return candidates
    raw = Path(nb_path)
    candidates.append(raw)
    if nb_path.startswith('/Workspace/'):
        candidates.append(Path(nb_path))
    else:
        candidates.append(Path('/Workspace') / nb_path.lstrip('/'))
    return candidates


def upward_candidates(start):
    start = Path(start).resolve()
    return [start, *start.parents]


def find_repo_root():
    starts = [Path.cwd(), *databricks_notebook_candidates()]
    seen = set()
    candidates = []
    for start in starts:
        try:
            for candidate in upward_candidates(start):
                key = str(candidate)
                if key not in seen:
                    seen.add(key)
                    candidates.append(candidate)
        except Exception:
            continue
    for candidate in candidates:
        if (candidate / 'pyproject.toml').exists() and (candidate / 'src' / 'pynnlf').exists():
            return candidate
    details = '\n'.join(str(c) for c in candidates[:30])
    raise FileNotFoundError('Could not locate the PyNNLF repo root. Checked:\n' + details)


REPO_ROOT = find_repo_root()
PROJECT_DIR = REPO_ROOT / 'publication' / 'journal_article_1'
if not PROJECT_DIR.exists():
    raise FileNotFoundError(f'Publication project folder not found: {PROJECT_DIR}')

os.chdir(PROJECT_DIR)
if str(REPO_ROOT / 'src') not in sys.path:
    sys.path.insert(0, str(REPO_ROOT / 'src'))

print(f'Repo root: {REPO_ROOT}')
print(f'Publication project: {PROJECT_DIR}')
print(f'Current working directory: {Path.cwd()}')


## 2. Databricks Dependency Install

On Databricks Environment v5 this should install/confirm the dependencies needed by all 12 models. Locally, this cell skips installation and uses your current Python environment.


In [ ]:
import importlib
import subprocess


def running_on_databricks():
    if 'DATABRICKS_RUNTIME_VERSION' in os.environ:
        return True
    try:
        dbutils  # noqa: F821
        return True
    except Exception:
        return False


def get_install_dependencies(default='yes'):
    choices = ['yes', 'no']
    try:
        dbutils.widgets.dropdown('install_dependencies', default, choices, 'Install dependencies')  # noqa: F821
        value = dbutils.widgets.get('install_dependencies')  # noqa: F821
    except Exception:
        value = default
    if value not in choices:
        raise ValueError(f'Unsupported install_dependencies={value!r}; expected one of {choices}')
    return value


def pip_install(args):
    cmd = [sys.executable, '-m', 'pip', 'install', '--disable-pip-version-check', *args]
    print('Running:', ' '.join(str(part) for part in cmd))
    subprocess.check_call(cmd)


INSTALL_DEPENDENCIES = get_install_dependencies()
print(f'Install dependencies: {INSTALL_DEPENDENCIES}')

if running_on_databricks() and INSTALL_DEPENDENCIES == 'yes':
    print('Databricks environment detected. Installing PyNNLF from local repo and full model dependencies...')
    pip_install(['--ignore-requires-python', '--no-deps', '-e', str(REPO_ROOT)])
    deps = [
        'typing_extensions==4.12.2',
        'PyYAML==6.0.2',
        'dill==0.3.9',
        'numpy==2.1.1',
        'pandas==2.2.3',
        'matplotlib==3.9.2',
        'scikit-learn==1.6.0',
        'scipy==1.14.1',
        'statsmodels==0.14.4',
        'joblib==1.4.2',
        'xgboost==3.0.0',
        'torch==2.5.1',
        'prophet==1.1.6',
        'cmdstanpy==1.2.5',
        'holidays==0.72',
    ]
    pip_install(['--upgrade', *deps])
    importlib.invalidate_caches()
    print('Install step complete. If Databricks prompts for a Python restart, restart and rerun from the top.')
else:
    print('Skipping dependency installation.')


## 3. Load Batch Spec And PyNNLF Engine

This notebook deliberately uses `experiment_result_databricks`, not the default `experiment_result`, so the Databricks complete run remains separate from the partial local laptop run.


In [ ]:
import pandas as pd
import yaml
import pynnlf
from pynnlf.discovery import discover_dataset_path, discover_model_name
from pynnlf.engine import run_experiment_engine
from pynnlf.hyperparams import load_hyperparameters, get_hp

BATCH_PATH = PROJECT_DIR / 'specs' / 'sa_bess_44hh_batch.yaml'
CONFIG_PATH = PROJECT_DIR / 'specs' / 'pynnlf_config.yaml'
HYPERPARAMS_PATH = PROJECT_DIR / 'models' / 'hyperparameters.yaml'
MODELS_DIR = PROJECT_DIR / 'models'
DATA_DIR = PROJECT_DIR / 'data'
RESULTS_ROOT = PROJECT_DIR / 'experiment_result_databricks'
RESULTS_ROOT.mkdir(parents=True, exist_ok=True)

batch = yaml.safe_load(BATCH_PATH.read_text(encoding='utf-8'))
config = yaml.safe_load(CONFIG_PATH.read_text(encoding='utf-8'))
config.setdefault('plot', {})['enabled'] = False
forecast_horizon_minutes = {key: int(value) for key, value in config['forecast_horizons'].items()}
hparams = load_hyperparameters(HYPERPARAMS_PATH)

print(f'Batch spec: {BATCH_PATH}')
print(f'Databricks experiment result folder: {RESULTS_ROOT}')
print(batch)


## 4. Validate Input Datasets

All three clean 44-household SA BESS datasets must be available in the publication data folder.


In [ ]:
EXPECTED_FILES = {
    'ds22': 'ds22_sa_bess_44hh_pos_underlying_load_30min.csv',
    'ds23': 'ds23_sa_bess_44hh_pos_net_load_with_pv_30min.csv',
    'ds24': 'ds24_sa_bess_44hh_pos_net_load_with_pv_battery_30min.csv',
}

summaries = []
for dataset_id, filename in EXPECTED_FILES.items():
    path = DATA_DIR / filename
    if not path.exists():
        raise FileNotFoundError(path)
    frame = pd.read_csv(path, parse_dates=['datetime'])
    if frame['datetime'].duplicated().any():
        raise ValueError(f'{dataset_id} has duplicate timestamps')
    if frame['netload_kW'].isna().any():
        raise ValueError(f'{dataset_id} has missing netload_kW')
    summaries.append({
        'dataset_id': dataset_id,
        'filename': filename,
        'rows': len(frame),
        'start': frame['datetime'].min(),
        'end': frame['datetime'].max(),
        'mean_netload_kW': frame['netload_kW'].mean(),
        'min_netload_kW': frame['netload_kW'].min(),
        'max_netload_kW': frame['netload_kW'].max(),
    })

summary_df = pd.DataFrame(summaries)
display(summary_df)


## 5. Run Complete 36-Experiment Batch

The run is resumable. It skips only combinations already present in `experiment_result_databricks`, not combinations from the local laptop `experiment_result` folder.


In [ ]:
def completed_keys(results_root):
    keys = set()
    for result_file in sorted(Path(results_root).glob('E*/E*_a1_experiment_result.csv')):
        try:
            row = pd.read_csv(result_file, nrows=1).iloc[0]
            keys.add((
                str(row.get('dataset_no', '')),
                int(row.get('forecast_horizon_min')),
                str(row.get('model_no', '')),
                str(row.get('hyperparameter_no', '')),
            ))
        except Exception:
            continue
    return keys


done = completed_keys(RESULTS_ROOT)
total = len(batch['datasets']) * len(batch['forecast_horizons']) * len(batch['model_and_hp'])
run_index = 0

for dataset_id in batch['datasets']:
    dataset_path = discover_dataset_path(DATA_DIR, dataset_id)
    for forecast_horizon_id in batch['forecast_horizons']:
        horizon_minutes = forecast_horizon_minutes[forecast_horizon_id]
        for model_id, hp_no in batch['model_and_hp']:
            run_index += 1
            key = (str(dataset_id), horizon_minutes, str(model_id), str(hp_no))
            if key in done:
                print(f'[skip {run_index}/{total}] {dataset_id} {forecast_horizon_id} {model_id} {hp_no}')
                continue
            model_name = discover_model_name(MODELS_DIR, model_id)
            hp = get_hp(hparams, model_name, hp_no)
            print(f'[run {run_index}/{total}] {dataset_id} {forecast_horizon_id} {model_id} {hp_no} -> {model_name}')
            run_experiment_engine(
                dataset_path=dataset_path,
                forecast_horizon_min=horizon_minutes,
                model_name=model_name,
                hyperparameter_no=hp_no,
                hyperparameter=hp,
                output_dir=RESULTS_ROOT,
                models_dir=MODELS_DIR,
                config=config,
            )
            done.add(key)

recap = pynnlf.recap_experiments(
    RESULTS_ROOT,
    output_path=RESULTS_ROOT / 'a1_experiment_result.csv',
    return_df=True,
)
print(f'Recap written: {RESULTS_ROOT / "a1_experiment_result.csv"}')


## 6. Validate Databricks Complete Recap

This cell confirms that the Databricks output folder contains exactly the 36 required SA BESS 44hh experiment combinations.


In [ ]:
recap_path = RESULTS_ROOT / 'a1_experiment_result.csv'
if not recap_path.exists():
    raise FileNotFoundError(recap_path)

recap = pd.read_csv(recap_path)
expected = set()
for dataset_id in batch['datasets']:
    for forecast_horizon_id in batch['forecast_horizons']:
        horizon_minutes = forecast_horizon_minutes[forecast_horizon_id]
        for model_id, hp_no in batch['model_and_hp']:
            expected.add((str(dataset_id), horizon_minutes, str(model_id), str(hp_no)))

actual_rows = recap.loc[
    recap['dataset_no'].astype(str).isin([str(ds) for ds in batch['datasets']])
    & pd.to_numeric(recap['forecast_horizon_min'], errors='coerce').isin([forecast_horizon_minutes[fh] for fh in batch['forecast_horizons']])
].copy()
actual = set(zip(
    actual_rows['dataset_no'].astype(str),
    pd.to_numeric(actual_rows['forecast_horizon_min'], errors='coerce').astype(int),
    actual_rows['model_no'].astype(str),
    actual_rows['hyperparameter_no'].astype(str),
))
missing = sorted(expected - actual)
extra = sorted(actual - expected)
if missing or extra:
    raise ValueError(f'Databricks SA BESS 44hh recap mismatch. Missing={missing[:10]}, extra={extra[:10]}')
if actual_rows.shape[0] != len(expected):
    raise ValueError(f'Expected {len(expected)} rows, found {actual_rows.shape[0]}')

print(f'Validated complete Databricks run: {actual_rows.shape[0]} / {len(expected)} rows')
display(actual_rows[['experiment_no', 'dataset_no', 'forecast_horizon_min', 'model_name', 'runtime_ms', 'test_nRMSE', 'test_nRMSE_stddev']].sort_values(['dataset_no', 'model_name']))
